In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 경고(W) 메시지만 숨김

In [2]:
import os 
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [2]:
import tensorflow as tf

# GPU 메모리 동적 할당 설정
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU 메모리 동적 할당 활성화 완료")
    except RuntimeError as e:
        print(e)

GPU 메모리 동적 할당 활성화 완료


In [3]:
tf.config.set_visible_devices([], 'GPU')

In [23]:
import pandas as pd

# KOTE 데이터셋 로드
train_df = pd.read_csv("KOTE/train.tsv", sep='\t')
test_df = pd.read_csv("KOTE/test.tsv", sep='\t')
val_df = pd.read_csv("KOTE/val.tsv", sep='\t')

# 데이터 확인
print(f"Train 데이터 크기: {train_df.shape}")
print(f"Test 데이터 크기: {test_df.shape}")
print(f"Validation 데이터 크기: {val_df.shape}")

# 데이터 샘플 출력
train_df.head()

Train 데이터 크기: (39999, 3)
Test 데이터 크기: (4999, 3)
Validation 데이터 크기: (4999, 3)


,39087,내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.,"2,13,15,16,29,39"
0,30893,"정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요...","0,5,7,10,19,22,29,35,36,38"
1,45278,"새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수...","1,2,7"
2,16398,미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ,"9,15,20,23,26,28,29"
3,13653,네 맞습니다 플스는 역시 30프레임이 어울리죠 ㅎ,"1,2,8,9,11,13,15,16,28,29,32,40,42"
4,13748,어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋㅋㅋ,"2,15,23,24,25,28,33"


In [28]:
import pandas as pd

# 올바른 컬럼명 지정 후 데이터 로드
train_df = pd.read_csv("KOTE/train.tsv", sep='\t', names=['id', 'comments', 'labels'], header=None)
test_df = pd.read_csv("KOTE/test.tsv", sep='\t', names=['id', 'comments', 'labels'], header=None)
val_df = pd.read_csv("KOTE/val.tsv", sep='\t', names=['id', 'comments', 'labels'], header=None)

# 데이터 확인
print(train_df.head())
print("📌 데이터 컬럼 목록:", train_df.columns)

      id                                           comments  \
0  39087              내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.   
1  30893  정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요...   
2  45278  새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수...   
3  16398       미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ   
4  13653                        네 맞습니다 플스는 역시 30프레임이 어울리죠 ㅎ   

                               labels  
0                    2,13,15,16,29,39  
1          0,5,7,10,19,22,29,35,36,38  
2                               1,2,7  
3                 9,15,20,23,26,28,29  
4  1,2,8,9,11,13,15,16,28,29,32,40,42  
📌 데이터 컬럼 목록: Index(['id', 'comments', 'labels'], dtype='object')


In [29]:
import re

# 텍스트 정제 함수 (숫자 유지)
def clean_text(text):
    text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣ0-9\s]", "", text)  # 한글, 숫자, 공백 제외한 문자 제거
    text = re.sub(r"\s+", " ", text).strip()  # 연속된 공백 제거
    return text

# 데이터 정제 적용
train_df['clean_text'] = train_df['comments'].apply(clean_text)
test_df['clean_text'] = test_df['comments'].apply(clean_text)
val_df['clean_text'] = val_df['comments'].apply(clean_text)

# 결과 확인
train_df[['comments', 'clean_text']].head(10)

,comments,clean_text
0,내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.,내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네
1,"정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요...",정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ...
2,"새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수...",새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함...
3,미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ,미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ
4,네 맞습니다 플스는 역시 30프레임이 어울리죠 ㅎ,네 맞습니다 플스는 역시 30프레임이 어울리죠 ㅎ
5,어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋㅋㅋ,어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋㅋㅋ
6,물갈이약 구매했어요...미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는...,물갈이약 구매했어요미미네에서 수조2개랑 여러가지 용품사면서 34번 주문했었는데 그 ...
7,"십일조는 겨우60인데? 나머지 다 어디감? 혹시 엄마아빠 그랜져따로타고, 외식자주하...",십일조는 겨우60인데 나머지 다 어디감 혹시 엄마아빠 그랜져따로타고 외식자주하는거아...
8,90년 줘야 하는거 아닌가? 쓰레기같은 것들.,90년 줘야 하는거 아닌가 쓰레기같은 것들
9,아주 부착성도 좋고 효과 100점 입니다,아주 부착성도 좋고 효과 100점 입니다


In [30]:
# ✅ 텍스트 데이터셋 로드
train_texts = train_df['clean_text'].tolist()
test_texts = test_df['clean_text'].tolist()
val_texts = val_df['clean_text'].tolist()

In [31]:
# 감정 라벨 개수 확인
print(train_df['labels'].value_counts())

labels
0,6,10,22,23,33                       125
0,6,10,12,22,23,33                    108
24                                     83
2,28,40,42                             65
0,10,22,37                             55
                                     ... 
0,3,10,12,20,21,22,23,24,27,31,35       1
0,5,10,19,20,22,25,27,31,36,38          1
0,2,3,8,20,22,23,24,28,29,32,33,39      1
1,11,14,15,16,27,29,38,39,41            1
2,9,10,15,18,23,33,35,39                1
Name: count, Length: 29332, dtype: int64


In [32]:
import pandas as pd

# 감정을 7개 그룹으로 정리하는 매핑 (KOTE의 44개 감정을 7개 감정 그룹으로 변환)
emotion_mapping = {
    '기쁨': [42, 40, 28],  # 기쁨, 행복, 즐거움/신남
    '슬픔': [5, 19, 25, 36],  # 슬픔, 절망, 패배/자기혐오, 서러움
    '놀람': [39, 34, 15, 2],  # 놀람, 경악, 신기함/관심, 감동/감탄
    '분노': [6, 22, 0],  # 화남/분노, 짜증, 불평/불만
    '공포': [18, 41],  # 공포/무서움, 불안/걱정
    '혐오': [31, 21],  # 증오/혐오, 역겨움/징그러움
    '중립': [24, 14, 43]  # 없음, 편안/쾌적, 안심/신뢰
}

# FER2013 기준 감정 라벨로 변환
fer2013_label_mapping = {
    '기쁨': 3,  # Happy
    '슬픔': 5,  # Sad
    '놀람': 6,  # Surprise
    '분노': 0,  # Angry
    '공포': 2,  # Fear
    '혐오': 1,  # Disgust
    '중립': 4   # Neutral
}

# 감정 그룹을 FER2013 기준으로 매핑하는 함수
def map_to_fer2013(label_str):
    if pd.isna(label_str) or label_str == '':
        return 6  # NaN 값은 중립(6)으로 처리

    label_list = list(map(int, label_str.split(',')))  # 숫자 리스트 변환
    for num in label_list:
        for key, values in emotion_mapping.items():  # KOTE 감정 그룹 확인
            if num in values:
                return fer2013_label_mapping[key]  # FER2013 감정으로 변환
    return 6  # 변환되지 않는 경우, 중립(6)으로 처리

# 감정 라벨 변환 적용 (NaN 값 처리 추가)
train_df['fer2013_emotion_grouped'] = train_df['labels'].fillna('').apply(map_to_fer2013)
test_df['fer2013_emotion_grouped'] = test_df['labels'].fillna('').apply(map_to_fer2013)
val_df['fer2013_emotion_grouped'] = val_df['labels'].fillna('').apply(map_to_fer2013)

# 변환된 감정 개수 확인
print(train_df['fer2013_emotion_grouped'].value_counts())

fer2013_emotion_grouped
0    18408
6    13984
4     3049
5     2300
3     1343
2      689
1      227
Name: count, dtype: int64


In [33]:
# ✅ 감정 라벨 (FER2013 매핑된 라벨)
train_labels = train_df['fer2013_emotion_grouped'].values
test_labels = test_df['fer2013_emotion_grouped'].values
val_labels = val_df['fer2013_emotion_grouped'].values

# 변환된 라벨 확인
print("라벨 변환 결과:", train_labels[:5])

라벨 변환 결과: [6 0 6 6 6]


In [34]:
# KoNLPy 라이브러리 설치
!pip install konlpy

from konlpy.tag import Okt

# Okt 토크나이저 인스턴스 생성
okt = Okt()

# 예제 문장 토큰화
text = "안녕하세요. 챗봇을 만들기 위한 텍스트 데이터 셋을 준비 중입니다."
tokens = okt.morphs(text)
print("형태소 단위 토큰:", tokens)



Defaulting to user installation because normal site-packages is not writeable
형태소 단위 토큰: ['안녕하세요', '.', '챗봇', '을', '만들기', '위', '한', '텍스트', '데이터', '셋', '을', '준비', '중', '입니다', '.']


In [35]:
from konlpy.tag import Okt

# Okt 토크나이저 인스턴스 생성
okt = Okt()

# 토큰화 함수
def tokenize(text):
    return okt.morphs(text)

# 토큰화 적용
train_df['tokenized'] = train_df['clean_text'].apply(tokenize)
test_df['tokenized'] = test_df['clean_text'].apply(tokenize)
val_df['tokenized'] = val_df['clean_text'].apply(tokenize)

# 결과 확인
print(train_df[['clean_text', 'tokenized']].head())

                                          clean_text  \
0                  내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네   
1  정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ...   
2  새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함...   
3       미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ   
4                        네 맞습니다 플스는 역시 30프레임이 어울리죠 ㅎ   

                                           tokenized  
0  [내, 가, 톰, 행크스, 를, 좋아하긴, 했나보다, 초기, 영화, 빼고는, 다, 봤네]  
1  [정말, 상상, 을, 초월, 하는, 무, 개념, 진상, 들, 상대, 하다, 우울증,...  
2  [새로운, 세상, 과, 조우, 한, 자의, 어린아이, 같은, 반응, 어쩌면, 회복,...  
3  [미역, 은, 원생생물계, 산호초, 는, 동물, ㅇㅇ, 아, 미역, 이, 바다, 의...  
4          [네, 맞습니다, 플스, 는, 역시, 30, 프레임, 이, 어울리죠, ㅎ]  


In [36]:
# 불용어 리스트
# "의", "가", "이", "은", "들", "는", "좀", "잘", "걍", "과",
#   "도", "를", "으로", "자", "에", "와", "한", "하다", "에서",
#    "까지", "부터", "마다", "보다", "더", "만", "요", "그리고",
#    "그러나", "하지만", "또한", "때문에", "그래서", "무엇", "어디",
#    "왜", "어떻게", "그래도", "그런데", "그러면", "하면", "이다",
#    "이런", "저런", "뿐", "만큼", "정도"
stopwords = [
    "의", "가", "이", "은", "들", "는", "걍", "과",
    "를", "으로", "자", "에", "와", "한", "하다", "에서",
    "요", "무엇", "어디", "하면", "이다"
]

# 불용어 제거 함수
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stopwords]

# 불용어 제거 적용
train_df['tokenized'] = train_df['tokenized'].apply(remove_stopwords)
test_df['tokenized'] = test_df['tokenized'].apply(remove_stopwords)
val_df['tokenized'] = val_df['tokenized'].apply(remove_stopwords)

# 결과 확인
print(train_df[['clean_text', 'tokenized']].head())

                                          clean_text  \
0                  내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네   
1  정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ...   
2  새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함...   
3       미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ   
4                        네 맞습니다 플스는 역시 30프레임이 어울리죠 ㅎ   

                                           tokenized  
0        [내, 톰, 행크스, 좋아하긴, 했나보다, 초기, 영화, 빼고는, 다, 봤네]  
1  [정말, 상상, 을, 초월, 하는, 무, 개념, 진상, 상대, 우울증, 공항, 장애...  
2  [새로운, 세상, 조우, 자의, 어린아이, 같은, 반응, 어쩌면, 회복, 된, 것,...  
3  [미역, 원생생물계, 산호초, 동물, ㅇㅇ, 아, 미역, 바다, 새, ㄱㅇㄱㅋㅋㅋㅋ...  
4                [네, 맞습니다, 플스, 역시, 30, 프레임, 어울리죠, ㅎ]  


In [43]:
# PyTorch 데이터셋 변환 코드 시작
import torch
from torch.utils.data import TensorDataset, DataLoader
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB_SIZE = 50000

# 단어 사전 생성
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

# 단어를 숫자로 변환
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)
val_sequences = tokenizer.texts_to_sequences(val_texts)

# ✅ 텍스트 토큰화 및 패딩
MAX_SEQUENCE_LENGTH = max(len(seq) for seq in train_sequences)

# 최대 문장 길이 설정 (패딩)
train_padded = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post')
test_padded = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post')
val_padded = pad_sequences(val_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

# NumPy 배열을 PyTorch 텐서로 변환 (정수 시퀀스는 long 타입으로)
train_text_tensor = torch.tensor(train_padded, dtype=torch.long)
train_label_tensor = torch.tensor(train_labels, dtype=torch.long)

test_text_tensor = torch.tensor(test_padded, dtype=torch.long)
test_label_tensor = torch.tensor(test_labels, dtype=torch.long)

val_text_tensor = torch.tensor(val_padded, dtype=torch.long)
val_label_tensor = torch.tensor(val_labels, dtype=torch.long)

# TensorDataset 생성: (텍스트, 레이블)
train_dataset = TensorDataset(train_text_tensor, train_label_tensor)
test_dataset = TensorDataset(test_text_tensor, test_label_tensor)
val_dataset = TensorDataset(val_text_tensor, val_label_tensor)

# DataLoader 생성
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train dataset size:", train_text_tensor.shape)
print("✅ PyTorch 텍스트 데이터셋 준비 완료!")

Train dataset size: torch.Size([40000, 98])
✅ PyTorch 텍스트 데이터셋 준비 완료!


In [44]:
import numpy as np

# 넘파이 배열 변환 
X_train, y_train = np.array(train_padded), np.array(train_labels)
X_val, y_val = np.array(val_padded), np.array(val_labels)
X_test, y_test = np.array(test_padded), np.array(test_labels)

print("훈련 데이터 크기:", X_train.shape, y_train.shape) 
print("검증 데이터 크기:", X_val.shape, y_val.shape)
print("테스트 데이터 크기:", X_test.shape, y_test.shape)

훈련 데이터 크기: (40000, 98) (40000,)
검증 데이터 크기: (5000, 98) (5000,)
테스트 데이터 크기: (5000, 98) (5000,)


In [ ]:
태양이형 수정 후

In [45]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight

# ─── 하이퍼파라미터 정의 ──────────────────────────────────────────
MAX_VOCAB_SIZE = 50000       # 토크나이저에서 지정한 num_words 값과 동일하게
EMBEDDING_DIM  = 300         # Keras 코드에서 쓰던 임베딩 차원


# ─── 1. 재현 가능한 결과를 위한 시드 고정 ─────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ─── 2. Attention 모듈 정의 ───────────────────────────────────────────────────────────────
class AttentionLayer(nn.Module):
    def __init__(self, feature_dim, dropout_rate=0.1):
        super().__init__()
        self.W = nn.Linear(feature_dim, feature_dim)
        self.V = nn.Linear(feature_dim, 1)
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, x, mask=None):
        # x: (batch, seq_len, feature_dim)
        score = self.V(torch.tanh(self.W(x)))  # (batch, seq_len, 1)
        if mask is not None:
            # mask: (batch, seq_len) -> (batch, seq_len, 1)
            score = score.masked_fill(mask.unsqueeze(-1)==0, -1e9)
        alpha = F.softmax(score, dim=1)         # (batch, seq_len, 1)
        alpha = self.dropout(alpha)
        context = (alpha * x).sum(dim=1)        # (batch, feature_dim)
        return context

# ─── 3. 모델 정의 ─────────────────────────────────────────────────────────────────────
class EmotionModel(nn.Module):
    def __init__(self,
                 vocab_size,
                 embedding_dim=300,
                 num_filters=64,
                 kernel_size=3,
                 lstm_units=128,
                 dropout_rate=0.3,
                 num_classes=7):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Conv1D expects (batch, channels, seq_len)
        self.conv = nn.Conv1d(in_channels=embedding_dim,
                              out_channels=num_filters,
                              kernel_size=kernel_size,
                              padding=kernel_size//2)
        self.bn   = nn.BatchNorm1d(num_filters)
        
        self.bilstm = nn.LSTM(input_size=num_filters,
                              hidden_size=lstm_units,
                              batch_first=True,
                              bidirectional=True)
        
        self.attn = AttentionLayer(feature_dim=2*lstm_units,
                                   dropout_rate=dropout_rate)
        
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1     = nn.Linear(2*lstm_units, 64)
        self.fc2     = nn.Linear(64, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)                          # (batch, seq_len, emb_dim)
        conv_in = emb.permute(0,2,1)                     # (batch, emb_dim, seq_len)
        conv_out = F.relu(self.bn(self.conv(conv_in)))  # (batch, num_filters, seq_len)
        seq = conv_out.permute(0,2,1)                   # (batch, seq_len, num_filters)
        
        lstm_out, _ = self.bilstm(seq)                  # (batch, seq_len, 2*lstm_units)
        attn_out    = self.attn(lstm_out)               # (batch, 2*lstm_units)
        
        x = self.dropout(attn_out)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)                              # (batch, num_classes)

# ─── 4. 학습/평가 함수 정의 ─────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device, max_grad_norm=1.0):
    model.train()
    total_loss, total_correct, total_count = 0, 0, 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(texts)
        loss = criterion(logits, labels)
        loss.backward()
        
        # ← 여기서 gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        
        optimizer.step()
        
        total_loss   += loss.item() * texts.size(0)
        preds         = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_count   += texts.size(0)
    return total_loss/total_count, total_correct/total_count


def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_count = 0, 0, 0
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            logits = model(texts)
            loss = criterion(logits, labels)
            
            total_loss += loss.item() * texts.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_count   += texts.size(0)
    return total_loss/total_count, total_correct/total_count

# ─── 5. 하이퍼파라미터 및 준비 ────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 주어진 DataLoader: train_loader, val_loader, test_loader
# class weight 계산 (train_loader.dataset.tensors[1] 이 레이블 텐서라고 가정)
all_train_labels = train_loader.dataset.tensors[1].cpu().numpy()
classes = np.unique(all_train_labels)
weights = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weights = torch.tensor(weights, dtype=torch.float, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

best_params = {
    'num_filters': 64,
    'kernel_size': 3,
    'lstm_units': 128,
    'dropout_rate': 0.3,
    'learning_rate': 3e-4,
    'epochs': 50,
}
n_ensemble = 7
early_stop_patience = 4

# ─── 6. Ensemble 학습 루프 ──────────────────────────────────────────────────────────────
ensemble_models = []
val_accuracies   = []

for i in range(n_ensemble):
    print(f"\n=== Training model {i+1}/{n_ensemble} (seed={42+i}) ===")
    set_seed(42 + i)
    
    model = EmotionModel(
        vocab_size=MAX_VOCAB_SIZE,
        embedding_dim=EMBEDDING_DIM,
        num_filters=best_params['num_filters'],
        kernel_size=best_params['kernel_size'],
        lstm_units=best_params['lstm_units'],
        dropout_rate=best_params['dropout_rate'],
        num_classes=7
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=best_params['learning_rate'])
    # train_one_epoch 호출 시 max_grad_norm 전달
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device, max_grad_norm=1.0
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2,
        verbose=True
    )    
    
    best_val_loss = float('inf')
    no_improve    = 0
    
    for epoch in range(1, best_params['epochs']+1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = eval_one_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        
        print(f"Epoch {epoch:02d} ▶ train_loss={train_loss:.4f}, train_acc={train_acc:.4f} │ "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
        
        # EarlyStopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            # 개선 시 체크포인트 저장
            torch.save(model.state_dict(), f"checkpoint_model_{i}.pt")
        else:
            no_improve += 1
            if no_improve >= early_stop_patience:
                print(f"--> Early stopping at epoch {epoch}")
                break
    
    # 체크포인트 로드 후 평가
    model.load_state_dict(torch.load(f"checkpoint_model_{i}.pt", map_location=device))
    _, val_acc = eval_one_epoch(model, val_loader, criterion, device)
    ensemble_models.append(model)
    val_accuracies.append(val_acc)

# ─── 7. 최적 모델 선택 및 저장 ─────────────────────────────────────────────────────────
best_idx = int(np.argmax(val_accuracies))
best_model = ensemble_models[best_idx]
print(f"\n🏆 Best model: #{best_idx+1} │ val_accuracy = {val_accuracies[best_idx]:.4f}")

os.makedirs("saved_models", exist_ok=True)
torch.save(best_model.state_dict(), f"saved_models/best_text_model.pt")
print("✅ Best model saved to saved_models/best_text_model.pt")



=== Training model 1/7 (seed=42) ===


/home/kangnengyee/.local/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 01 ▶ train_loss=1.8640, train_acc=0.3500 │ val_loss=1.8311, val_acc=0.4300
Epoch 02 ▶ train_loss=1.7625, train_acc=0.3806 │ val_loss=1.8292, val_acc=0.2458
Epoch 03 ▶ train_loss=1.5989, train_acc=0.4014 │ val_loss=1.8733, val_acc=0.2784
Epoch 04 ▶ train_loss=1.4560, train_acc=0.4472 │ val_loss=1.9581, val_acc=0.3702
Epoch 05 ▶ train_loss=1.2683, train_acc=0.4995 │ val_loss=2.1566, val_acc=0.4132
Epoch 06 ▶ train_loss=1.0433, train_acc=0.5672 │ val_loss=2.3563, val_acc=0.3760
--> Early stopping at epoch 6


/tmp/ipykernel_2723/2726209103.py:205: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"checkpoint_model_{i}.pt", map_location=device))



=== Training model 2/7 (seed=43) ===
Epoch 01 ▶ train_loss=1.8500, train_acc=0.3602 │ val_loss=1.8323, val_acc=0.4464
Epoch 02 ▶ train_loss=1.7453, train_acc=0.3947 │ val_loss=1.8365, val_acc=0.3064
Epoch 03 ▶ train_loss=1.6227, train_acc=0.3997 │ val_loss=1.9024, val_acc=0.3658
Epoch 04 ▶ train_loss=1.4700, train_acc=0.4289 │ val_loss=1.9555, val_acc=0.4442
Epoch 05 ▶ train_loss=1.2447, train_acc=0.4996 │ val_loss=2.1715, val_acc=0.3816
--> Early stopping at epoch 5

=== Training model 3/7 (seed=44) ===
Epoch 01 ▶ train_loss=1.8527, train_acc=0.3892 │ val_loss=1.8319, val_acc=0.4046
Epoch 02 ▶ train_loss=1.7574, train_acc=0.4172 │ val_loss=1.8151, val_acc=0.4064
Epoch 03 ▶ train_loss=1.6391, train_acc=0.4226 │ val_loss=1.8625, val_acc=0.4250
Epoch 04 ▶ train_loss=1.4846, train_acc=0.4586 │ val_loss=1.9696, val_acc=0.3890
Epoch 05 ▶ train_loss=1.3018, train_acc=0.4997 │ val_loss=2.1066, val_acc=0.3760
Epoch 06 ▶ train_loss=1.0836, train_acc=0.5646 │ val_loss=2.3949, val_acc=0.4114
-->